In [2]:
# ============================================================
# FULL PIPELINE FOR LOBSTER BIOACOUSTICS CLASSIFICATION
# REFINED VERSION: MEMORY-OPTIMIZED & MULTI-FEATURE
# ============================================================

import os
import sys
import numpy as np
import librosa
import pandas as pd

# GPU Memory Management
import tensorflow as tf
from tensorflow.keras import backend as K

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled")
    except RuntimeError as e:
        print(f"Memory growth error: {e}")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense

# ============================================================
# DATASET ROOT
# ============================================================
BASE_DATASET_DIR = "/home/feliciano/LOBSTER SOUNDS/DatasetClass"

# ============================================================
# REFINED FEATURE EXTRACTION
# ============================================================

def extract_refined_features(audio, sr):
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    mfcc_mean = np.mean(mfcc, axis=1)
    
    centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)
    centroid_mean = np.mean(centroid, axis=1)
    
    contrast = librosa.feature.spectral_contrast(y=audio, sr=sr)
    contrast_mean = np.mean(contrast, axis=1)
    
    rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr)
    rolloff_mean = np.mean(rolloff, axis=1)
    
    zcr = librosa.feature.zero_crossing_rate(y=audio)
    zcr_mean = np.mean(zcr, axis=1)
    
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)
    
    return np.concatenate([mfcc_mean, centroid_mean, contrast_mean, rolloff_mean, zcr_mean, chroma_mean])

def load_audio_folder(folder, label):
    X, y = [], []
    if not os.path.exists(folder):
        return np.array([]), np.array([])
    for f in os.listdir(folder):
        if f.lower().endswith(".wav"):
            try:
                audio, sr = librosa.load(os.path.join(folder, f), sr=22050)
                if len(audio) == 0: continue
                features = extract_refined_features(audio, sr)
                X.append(features)
                y.append(label)
            except:
                continue
    return np.array(X), np.array(y)

# Loading Data
X_adult, y_adult = load_audio_folder(os.path.join(BASE_DATASET_DIR, "female_lobsters"), 1)
X_juv, y_juv = load_audio_folder(os.path.join(BASE_DATASET_DIR, "male_lobsters"), 0)

X = np.vstack([X_adult, X_juv])
y = np.concatenate([y_adult, y_juv])

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================================================
# CLASSICAL MODELS
# ============================================================
results = []
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss"),
    "SVM": SVC(probability=True)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    yp = model.predict(X_test)
    yp_prob = model.predict_proba(X_test)[:, 1]
    results.append([name, f"{accuracy_score(y_test, yp)*100:.2f}", f"{roc_auc_score(y_test, yp_prob)*100:.2f}"])

# ============================================================
# DEEP LEARNING (WITH MEMORY CLEANUP)
# ============================================================
X_train_dl = X_train[..., np.newaxis]
X_test_dl = X_test[..., np.newaxis]

def build_cnn(depth, dilated=False):
    model = Sequential([
        Input(shape=(X_train.shape[1], 1))
    ])
    for i in range(depth):
        model.add(Conv1D(32, 3, dilation_rate=2 if dilated else 1, padding="same", activation="relu"))
        if i < depth - 1: model.add(MaxPooling1D(2))
    model.add(Flatten())
    model.add(Dense(64, activation="relu"))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer="adam", loss="binary_crossentropy")
    return model

# Loop with explicit memory clearing
for depth in [1, 2, 3]:
    for is_dilated in [False, True]:
        label = f"1D-{'D' if is_dilated else ''}CNN ({depth}L)"
        
        # Clear GPU memory before building a new model
        K.clear_session()
        
        try:
            nn = build_cnn(depth, dilated=is_dilated)
            # Reduced batch size to save VRAM
            nn.fit(X_train_dl, y_train, epochs=10, batch_size=16, verbose=0)
            
            prob = nn.predict(X_test_dl, batch_size=16).ravel()
            pred = (prob > 0.5).astype(int)
            
            results.append([label, f"{accuracy_score(y_test, pred)*100:.2f}", f"{roc_auc_score(y_test, prob)*100:.2f}"])
            
            # Delete model objects to free RAM/VRAM
            del nn
        except Exception as e:
            print(f"Skipping {label} due to memory: {e}")

# ============================================================
# OUTPUT
# ============================================================
df = pd.DataFrame(results, columns=["Model", "Acc (%)", "AUC (%)"])
print("\n", df.to_string(index=False))

Memory growth error: Physical devices cannot be modified after being initialized


/home/feliciano/anaconda3/lib/python3.12/site-packages/librosa/core/pitch.py:101: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
I0000 00:00:1777228738.719748   52350 cuda_dnn.cc:529] Loaded cuDNN version 90201
I0000 00:00:1777228739.500274   52350 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step

         Model Acc (%) AUC (%)
Random Forest   92.13   97.28
      XGBoost   94.05   98.59
          SVM   94.87   98.82
  1D-CNN (1L)   94.46   98.67
 1D-DCNN (1L)   95.01   98.91
  1D-CNN (2L)   94.80   98.74
 1D-DCNN (2L)   94.53   98.46
  1D-CNN (3L)   93.91   98.49
 1D-DCNN (3L)   93.84   98.41
